In [ ]:
#Installing the libraries

In [1]:
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.8 MB/s eta 0:00:00


In [ ]:
#Uploading pdf

In [2]:
from google.colab import files

uploaded = files.upload()

Saving sample.pdf to sample.pdf


In [ ]:
#Extracting Pdf text

In [3]:
from pypdf import PdfReader

reader = PdfReader("sample.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

CONTENTS
C
HAPTER
 1: Don’t Try
The Feedback Loop from Hell
The Subtle Art of Not Giving a Fuck
So Mark, What the Fuck Is the Point of This Book Anyway?
C
HAPTER
 2: Happiness Is a Problem
The Misadventures of Disappointment Panda
Happiness Comes from Solving Problems
Emotions Are Overrated
Choose Your Struggle
C
HAPTER
 3: You Are Not Special
Things Fall Apart
The Tyranny of Exceptionalism
B-b-b-but, If I’m Not Going to Be Special or Extraordinary, What’s the
Point?
C
HAPTER
 4: The Value of Suffering
The Self-Awareness Onion
Rock Star Problems
Shitty Values
Defining Good and Bad Values
C
HAPTER
 5: You Are Always Choosing
The Choice
The Responsibility/Fault Fallacy
Responding to Tragedy
Genetics and the Hand We’re Dealt
Victimhood Chic
There Is No “How”C
HAPTER
 6: You’re Wrong About Everything (But So Am I)
Architects of Our Own Beliefs
Be Careful What You Believe
The Dangers of Pure Certainty
Manson’s Law of Avoidance
Kill Yourself
How to Be a Little Less Certain of Yourself
C
HAPT

In [4]:
#Chunk text
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

chunks = chunk_text(text)

print("Chunks:", len(chunks))

Chunks: 698


In [5]:
#Generating embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(698, 384)


In [13]:

print(embeddings.shape)

(698, 384)


In [14]:
print(context)

ion and sympathy.
“Victimhood chic” is in style on both the right and the left today, among
both the rich and the poor. In fact, this 
may be the first time in human history
that every single demographic group has felt unfairly victimized
simultaneously. And they’re all riding the highs of the moral indignation that
comes along with it.
Right now, 
anyone 
who is offended about 
anything—
whether it’s the fact
that a book about racism was assigned in a university class, or that Christmas
trees w

become easier than ever to push responsibility—for even the tiniest of
infractions—onto some other group or person. In fact, this kind of public
blame/shame game has become popular; in certain crowds it’s even seen as
“cool.” The public sharing of “injustices” garners far more attention and
emotional outpouring than most other events on social media, rewarding
people who are able to perpetually feel victimized with ever-growing
amounts of attention and sympathy.
“Victimhood chic” is in style o

In [ ]:
#Creating FAISS index

In [6]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings)
)

print("FAISS index created")

FAISS index created


In [ ]:
#Gemini configuration

In [10]:
import google.generativeai as genai

genai.configure(
    api_key="GEMINI_API_KEY"
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [ ]:
#Chatbot loop

In [12]:
while True:

    query = input(
        "\nAsk Question (exit to stop): "
    )

    if query.lower() == "exit":
        break

    query_embedding = embedding_model.encode(
        [query]
    )

    D, I = index.search(
        np.array(query_embedding),
        k=3
    )

    retrieved_chunks = [
        chunks[idx]
        for idx in I[0]
    ]

    context = "\n".join(
        retrieved_chunks
    )

    prompt = f"""

    Answer only using the provided context.

    If the answer is not present,
    say:
    "I could not find the answer in the document."

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.generate_content(
        prompt
    )

    print("\nAnswer:")
    print(response.text)


Ask Question (exit to stop): feedback loop

Answer:
The Feedback Loop from Hell is a cycle where one feels bad about feeling bad. Examples include:
*   Lying about how much you’re worrying.
*   Feeling so guilty for every mistake that you begin to feel guilty about how guilty you’re feeling.
*   Getting sad and alone so often that it makes you feel even more sad and alone just thinking about it.

Ask Question (exit to stop): Victimhood Chic

Answer:
“Victimhood chic” is in style today on both the right and the left, among both the rich and the poor. It may be the first time in human history that every single demographic group has felt unfairly victimized simultaneously, and they are riding the highs of moral indignation that come along with it.

It has made it easier than ever to push responsibility for even the tiniest infractions onto some other group or person. This public blame/shame game has become popular, even seen as “cool” in certain crowds, and the public sharing of “injusti